# Instacart Market Basket Analysis

## Project overview

This notebook examines customer ordering behavior in the Instacart dataset. I move from structural inspection and data cleaning into exploratory analysis focused on order timing, basket size, product popularity, and reorder behavior. The goal is to produce results that are both analytically sound and useful in a business context, particularly for merchandising, retention, forecasting, and recommendation strategy.


## Portfolio note

This version of the notebook is prepared for GitHub and portfolio review. The analysis is preserved, but the notebook has been cleaned of reviewer comments, unfinished course prompts, and environment-specific file paths. The code now assumes the data files are stored locally in a `datasets/` folder, which keeps the project portable outside the original course environment.


In [ ]:
# Import the libraries used throughout the analysis
import os
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Load the dataset files from a local datasets/ folder
# The raw data is not included in this repository due to file size.
# To run this notebook, download the Instacart dataset separately and place the files in:
# datasets/orders.csv
# datasets/products.csv
# datasets/departments.csv
# datasets/aisles.csv
# datasets/order_products__prior.csv
# datasets/order_products__train.csv

DATA_PATH = "datasets"

orders = pd.read_csv(os.path.join(DATA_PATH, "orders.csv"))
products = pd.read_csv(os.path.join(DATA_PATH, "products.csv"))
departments = pd.read_csv(os.path.join(DATA_PATH, "departments.csv"))
aisles = pd.read_csv(os.path.join(DATA_PATH, "aisles.csv"))

order_products_prior = pd.read_csv(os.path.join(DATA_PATH, "order_products__prior.csv"))
order_products_train = pd.read_csv(os.path.join(DATA_PATH, "order_products__train.csv"))

order_products = pd.concat(
    [order_products_prior, order_products_train],
    ignore_index=True
)


## Initial data review

I begin by reviewing each dataset with `.head()` and `.info()` to understand its structure, data types, and completeness before making any cleaning decisions.

In [ ]:
# In this cell, type "orders" below this line and execute the cell
print(orders.head())

In [ ]:
# In this cell, type "products" below this line and execute the cell
print(products.head())

I repeat this review across the remaining tables to understand how the datasets relate to one another and to identify any issues that could affect downstream analysis.

In [ ]:
# In this cell, type "orders.info() below this line and execute the cell
print(orders.info(show_counts=True))

As I review the outputs, I pay close attention to non-null counts. Any column with fewer non-null values than expected warrants a closer look before I rely on it in analysis.

In [ ]:
# In this cell, run orders_products.info() below, but include the argument show_counts=True since this is a large file.
print(order_products.info(show_counts=True))

I continue the same structure check for the remaining tables so I can identify missing values early and confirm that the core data is ready for cleaning.

In [ ]:
# Review the structure of the remaining tables
print("-" * 84)
print("Products info")
products.info(show_counts=True)

print("-" * 84)
print("Departments info")
departments.info(show_counts=True)

print("-" * 84)
print("Aisles info")
aisles.info(show_counts=True)


## Missing-value review and treatment

After the initial inspection, I investigate each column with missing values and decide how to handle it based on context rather than applying a single blanket rule.


### Products table

I begin with the `products` table because missing product names would weaken any later product-level interpretation.

In [ ]:
# Display rows where the product_name column has missing values
print(products[products['product_name'].isna()])

The missing `product_name` values appear concentrated in rows associated with `aisle_id` 100 and `department_id` 21. I test that pattern directly to determine whether it is isolated or part of a broader issue.

In [ ]:
# Combine conditions to check for missing product names in aisles other than 100
oa_filter = (products['product_name'].isna()) & (products['aisle_id'] != 100)
other_aisles = products[oa_filter]
print(other_aisles)

In [ ]:
# Combine conditions to check for missing product names in departments other than 21
d_filter = (products['product_name'].isna()) & (products['department_id'] != 21)
other_departments = products[d_filter]
print(other_departments)

To interpret those rows more clearly, I look up the corresponding aisle and department labels in the supporting reference tables.

In [ ]:
# What is this aisle and department?
print(departments[departments['department_id'] == 21])
print(aisles[aisles['aisle_id'] == 100])

In [ ]:
# Fill missing product names with 'Unknown'
products['product_name'] = products['product_name'].fillna('Unknown')

### Orders table

I then review missing values in the `orders` table to determine whether they represent a data-quality problem or a normal part of customer behavior.

In [ ]:
# Display rows where the days_since_prior_order column has missing values
print(orders[orders['days_since_prior_order'].isna()])

In [ ]:
# Are there any missing values where it's not a customer's first order?
print(orders[(orders['days_since_prior_order'].isna()) & (orders['order_number'] != 1)])

All missing values in `days_since_prior_order` correspond to a customer’s first order. That pattern is expected, because a first order has no previous order to measure against. I leave those values as `NaN` so the column remains faithful to the underlying meaning of the data.

### Order-products table

I next examine missing values in `order_products`, where the issue affects the position at which items were added to the cart.

In [ ]:
# Display rows where the add_to_cart_order column has missing values
print(order_products[order_products['add_to_cart_order'].isna()])

In [ ]:
# Use .min() and .max() to find the minimum and maximum values for this column.
print(f"The minimum value for the add_to_cart_order column is {int(order_products['add_to_cart_order'].min())}.")
print()
print(f"The maximum value for this column is {int(order_products['add_to_cart_order'].max())}.")

In [ ]:
# Save all order IDs with at least one missing value in 'add_to_cart_order'
import numpy as np #Importing Numerical Python to use tools within.
orders_with_missing = np.unique(order_products[order_products['add_to_cart_order'].isna()]['order_id']) #Saving Order ID's with missing Add-To-Cart data to new variable.
print(f"There are {len(orders_with_missing)} order IDs with at least one missing value in the 'add_to_cart_order' column.") #Counts the number of orders with at least one missing value in 'add_to_cart_order'

In [ ]:
# Do all orders with missing values have more than 64 products?
filter_64 = ((order_products['order_id'].isin(orders_with_missing)) & (order_products['add_to_cart_order'] == 64))
orders_with_64th_item = order_products[filter_64]
print(orders_with_64th_item) #All 70 order IDs have at least 64 products added to the cart, meaning an error happend with the 'add_to_cart_order' values after 64. 

In [ ]:
# Replace missing values with 999 and convert column to integer type
order_products['add_to_cart_order'] = order_products['add_to_cart_order'].fillna(999)

The missing values in `add_to_cart_order` appear only for items placed 65th or later in the cart. I preserve those rows by assigning a placeholder value and keep that decision in mind when interpreting basket-position results later.

## Duplicate-value review

With missing values addressed, I check each dataset for duplicate records so the analysis is built on a cleaner and more reliable foundation.


### `orders` data frame

In [ ]:
# Find the number of duplicate rows in the orders dataframe
print(orders.duplicated().sum())

In [ ]:
# View the duplicate rows
print(orders[orders.duplicated()])

In [ ]:
# Remove duplicate orders
orders = orders.drop_duplicates().reset_index(drop=True)

In [ ]:
# Double check for duplicate rows
print(orders.duplicated().sum())

### Products table

In [ ]:
# Check for fully duplicate rows
print(products[products.duplicated()])

In [ ]:
# Check for just duplicate product IDs using subset='product_id' in duplicated()
print(products[products['product_id'].duplicated()])

To evaluate duplicate product names consistently, I convert them to lowercase so capitalization differences do not hide genuine duplicates.

In [ ]:
# Check for just duplicate product names (convert names to lowercase to compare better)
products['product_name'] = products['product_name'].str.lower()
print(products[products['product_name'].duplicated()])

I then inspect how those repeated names appear in the data so I can distinguish between repeated product names and truly duplicated records.

In [ ]:
products[products['product_name'].str.lower() == 'high performance energy drink']

In [ ]:
# Drop duplicate product names (case insensitive)
products['product_name'].drop_duplicates().reset_index(drop = True)

### Departments table

In [ ]:
# Check for duplicate entries in the departments dataframe
print(departments[departments.duplicated()])

### Aisles table

In [ ]:
# Check for duplicate entries in the aisles dataframe
print(aisles[aisles.duplicated()])

### Order-products table

In [ ]:
# Check for duplicate entries in the order_products dataframe
print(order_products[order_products.duplicated()])

With the main cleaning steps complete, I move into exploratory analysis to understand order timing, basket size, product popularity, and reorder behavior.

### Validating time fields

Before analyzing order patterns, I verify that `order_hour_of_day` and `order_dow` fall within sensible ranges.

In [ ]:
print('Orders hour of the day include:')
print(orders['order_hour_of_day'].unique())
print()
print('Orders day of the week include:')
print(orders['order_dow'].unique())

In [ ]:
print('Orders hour of the day (sorted):')
print(sorted(orders['order_hour_of_day'].unique()))
print()
print('Orders day of the week (sorted):')
print(sorted(orders['order_dow'].unique()))

### Shopping patterns by hour

I analyze `order_hour_of_day` to identify when customers are most active and to highlight the daily rhythm of grocery shopping.

In [ ]:
from matplotlib import pyplot as plt

hourly_orders = orders['order_hour_of_day'].value_counts().sort_index()

hourly_orders.plot(title = 'Time of Day People Shop for Groceries',
                  xlabel = 'Hour in the Day',
                  ylabel = 'Volume of Orders',
                  style = 'o-',
                  grid = True,
                   xticks = range(0, 23, 1)
                  )

plt.show()

#### Analysis

- Shopping activity is concentrated during the day rather than overnight.
- The strongest ordering window appears around late morning, with additional strength into the afternoon.
- This pattern is useful for thinking about operational load, staffing, fulfillment, and promotional timing.


Most orders occur between 9:00 AM and 5:00 PM, with especially strong activity around 10:00 AM and 3:00 PM.

### Shopping patterns by day of week

I analyze `order_dow` to compare order volume across the week and determine whether customer activity clusters around particular days.

In [ ]:
orders_per_day = orders['order_dow'].value_counts().sort_index()

orders_per_day.plot(title = 'Days People Shop for Groceries',
                   ylabel = 'Volume of Orders',
                   xlabel = 'Day of the Week',
                   style = 'o-',
                    grid = True,
                    xticks = range(0, 6, 1)
                   )

plt.show()

#### Analysis

- Ordering is strongest at the start of the weekly cycle.
- Activity softens through the middle of the week and then begins to recover.
- Even without a fully labeled weekday dictionary, the weekly pattern is strong enough to suggest recurring customer routines.


The data dictionary does not explicitly map each integer to a weekday, so I interpret the results cautiously. Assuming Sunday corresponds to `0`, the highest ordering volume occurs on Sunday and Monday.

### Time between orders

I examine `days_since_prior_order` to understand how frequently customers return to place another order.

In [ ]:
days_until_reordering = orders['days_since_prior_order'].value_counts().sort_index()

days_until_reordering.plot(title = 'Days Until Placing Another Order',
                          ylabel = 'Order Volume',
                          xlabel = 'Days Elapsed Since Last Order',
                          grid = True,
                          xticks = range(0, 30, 2),
                          rot = 45)

plt.show()

#### Analysis

- Seven days is the clearest mode, which points to a strong weekly purchasing rhythm.
- Additional spikes at 14, 21, and 28 days suggest that many customers follow repeating grocery cycles.
- The sharp concentration at 30 days may reflect either genuine monthly behavior or the way the dataset was recorded.


The `0` values likely represent customers who placed more than one order on the same day.

Setting aside the spike at 30 days, many customers return within 2 to 10 days, with 7 days as the most common interval.

### Comparing Wednesday and Saturday order patterns

I compare hourly order distributions for Wednesday and Saturday to see whether customer behavior differs between a weekday and a weekend day.

In [ ]:
wednesday_orders = orders[orders['order_dow'] == 3]['order_hour_of_day'].value_counts().sort_index()
saturday_orders = orders[orders['order_dow'] == 6]['order_hour_of_day'].value_counts().sort_index()

In [ ]:
wednesday_and_saturday = pd.concat([wednesday_orders, saturday_orders], axis = 'columns')

In [ ]:
wednesday_and_saturday.columns = ['wednesday_orders', 'saturday_orders']

In [ ]:
wednesday_and_saturday.plot(title = 'Orders on Wednesdays and Saturdays',
                           xlabel = 'Hour of the Day',
                           ylabel = 'Volume of Orders',
                           grid = True,
                           xticks = range(0, 24, 1),
                            yticks = range(0, 5500, 500),
                           kind = 'bar')
plt.show()

#### Analysis

- Both days show their strongest activity during normal daytime shopping hours.
- Wednesday includes a modest midday dip that is much less visible on Saturday.
- That difference likely reflects weekday work schedules or routine constraints that are less pronounced on weekends.


Wednesday shows a noticeable dip from 11:00 to 13:00 that does not appear on Saturday. That difference may reflect weekday work patterns or lunchtime routines.

### Orders per customer

I examine how many orders each customer has placed to understand how concentrated repeat activity is across the user base.

In [ ]:
orders_per_customer = orders.groupby('user_id')['order_id'].count().sort_values()

In [ ]:
orders_per_customer.plot(kind = 'hist',
                        bins = 16,
                        xticks = range(1, 17, 1),
                        range = (0, 16),
                        yticks = range(0, 57000, 4000),
                        title = 'Orders Per Customer',
                        grid = True,
                        ylim = (0, 57000))

plt.xlabel('Number of Orders')
plt.ylabel('Number of Customers')

plt.show()

#### Analysis

- Most customers have placed relatively few orders.
- The distribution drops off quickly as order count rises, which indicates that a smaller segment of heavy users accounts for a disproportionate share of repeat activity.
- This pattern matters for retention analysis, loyalty strategy, and lifecycle targeting.


Most customers in the dataset have placed between 1 and 10 orders, and the number of customers declines sharply as total order count rises.

### Most popular products

To identify the most frequently purchased items, I merge product metadata into the order data and rank products by total purchase count.

In [ ]:
orders_and_products = order_products.merge(products,
                                          left_on = 'product_id',
                                          right_on = 'product_id')

In [ ]:
grouped_products = orders_and_products.groupby(['product_id', 'product_name']).size()
grouped_products = grouped_products.sort_values(ascending=False)
top_20 = grouped_products.head(20)
print(top_20)

In [ ]:
top_20.plot(title = 'Top 20 Popular Products', 
            ylabel = 'Number of Orders', 
            xlabel = 'Products',
            figsize = (12, 6),
            kind = 'bar')

plt.show()

#### Analysis

- The most frequently purchased items are dominated by produce.
- Milk stands out as one of the few high-volume staples outside produce.
- The overall ranking points to strong recurring demand for fresh, everyday essentials.


The top 20 products are overwhelmingly produce, with milk as a notable exception.

### Items per order

I calculate how many products appear in each order to understand typical basket size and the overall shape of the distribution.

In [ ]:
group_ops = order_products.groupby('order_id')
number_per_order = group_ops['product_id'].count()

In [ ]:
items_per_order = number_per_order.value_counts().sort_index()

#Added (Before Filtering) to make note that this is not the final visualization. The next will focus on the majority of the data.
items_per_order.plot(kind = 'bar',
                    title = 'Items Per Order (Before Filtering)',
                    xticks = range(0, 50, 5),
                    xlabel = 'Number of Items',
                    ylabel = 'Volume of Orders')

plt.show()


The distribution has a long tail, so I narrow the view to the lower range in order to examine the most common basket sizes more clearly.

In [ ]:
filtered_ipo = items_per_order[items_per_order.index <= 35] #Creating a filtered version of 'items_per_order'
filtered_ipo.plot(kind = 'bar',
                  title = 'Items Per Order',
                  xlabel = 'Number of Items',
                  ylabel = 'Volume of Orders')

plt.show()

#### Analysis

- The most typical order contains about 5 or 6 items.
- Most orders are relatively small, even though a long tail of larger baskets remains present.
- This distribution is useful context for merchandising, cart design, and recommendation placement.


A typical order contains about 5 or 6 items, and most orders fall between 1 and 20 items.

### Most frequently reordered products

I isolate rows where `reordered == 1`, merge in product names, and rank products by how often they return to customers’ carts.

In [ ]:
reordered_products = order_products[order_products['reordered'] == 1] #Filtering the DataFrame to only reordered products

In [ ]:
#Combining 'reordered_products' and 'products' DataFrames on 'product_id'. This variable 'rp_combo' stands for reordered_products combination.
rp_combo = reordered_products.merge(products,
                                   left_on = 'product_id',
                                   right_on = 'product_id')

rp_combo = rp_combo.groupby(['product_id', 'product_name']).size() #Calculating how many times each product was reordered.

In [ ]:
rp_combo = rp_combo.sort_values(ascending = False) #Sorting values in descending order.

rp_top_20 = rp_combo.head(20) #Filtering down to the top 20 most reordered items and storing them into a 'rp_top_20' variable.

In [ ]:
 

rp_top_20.plot(title = 'Top 20 Most Reordered Items',
              kind = 'bar',
              xlabel = 'Items',
              ylabel = 'Frequency')

plt.show()

#### Analysis

- Produce and dairy dominate the list of most frequently reordered products.
- The pattern is consistent with habitual replenishment of perishable staples.
- These products are natural candidates for replenishment prompts, recommendations, and demand-planning use cases.


Produce and dairy account for many of the most frequently reordered products, which fits the expectation that perishable staples are purchased repeatedly.

### Reorder proportion by product

I then shift from raw reorder counts to reorder rates by measuring what share of each product’s orders are reorders rather than first-time purchases.

In [ ]:
# I used the variable 'rp_per_item' to stand for 'Reorder Proportions Per Item', since this is what we are aiming to narrow in on.
rp_per_item = order_products.merge(products) #Merged 'order_products' and 'products' datasets.
rp_per_item = rp_per_item.groupby(['product_id', 'product_name']) #Isolated each product's order history
reorder_proportions = rp_per_item['reordered'].mean() #Here, I calculated teh reorder proportions per item.
sorted_rps = reorder_proportions.sort_values(ascending = False) #I used the new variable 'sorted_rps' to stand for 'Sorted Reorder Proportions'.
sorted_rps.reset_index() #Organized data into a readable DataFrame

In [ ]:
# Sample products across reorder-rate ranges to see what kinds of items appear at different levels
for low in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0]:
    high = low + 0.1
    if low == 0.9:
        subset = sorted_rps[(sorted_rps >= low) & (sorted_rps <= 1.0)]
    else:
        subset = sorted_rps[(sorted_rps >= low) & (sorted_rps < high)]

    print(f"\nReorder-rate range {low:.1f} to {high:.1f}:")
    if len(subset) == 0:
        print("No products in this range.")
    else:
        print(subset.sample(min(10, len(subset)), random_state=42))


#### Analysis

- Reorder rate adds a different perspective from raw popularity because it highlights repeat-buy behavior directly.
- Highly reordered products are often fast-moving staples, while lower-rate products may be more occasional or one-time purchases.
- This metric can support both recommendation systems and demand forecasting because it captures habitual behavior more clearly than volume alone.
